# Preparación de ligandos para acoplamiento molecular (docking)

**Conversión de un archivo CSV con secuencias SMILES a estructuras tridimensionales listas para Docking**

QFB. Luis Ángel Urdez Rojas

---

## Consideraciones previas

### 1. Formato del archivo CSV de entrada

El archivo debe contener dos columnas: `ID` y `SMILES`. Los encabezados son reconocidos independientemente de si usan mayúsculas o espacios adicionales.

```
ID,SMILES
5206,FCOC(C(F)(F)F)C(F)(F)F
122357,[H]C(=O)[C@H](O)[C@H](O)COP(O)(O)=O
3870203,OC1=CC(=CC(=C1O)[N+]([O-])=O)[N+]([O-])=O
```

* **ID**: Es un identificador único. Este valor permite mantener la trazabilidad de los datos, ya que el nombre de cada archivo final derivará de esta columna. En caso de existir identificadores repetidos, el sistema les asignará un sufijo numérico.
* **SMILES**: Contiene la representación en texto de la estructura química, con una molécula por fila. Cualquier columna adicional presente en el archivo será ignorada.

### 2. Configuración del entorno de Colab

**La GPU es opcional, se recomienda T4 de GoogleColab.** La curación, los filtros y la generación de geometría se ejecutan íntegramente en la unidad central de procesamiento (CPU) mediante RDKit. El único paso que puede aprovechar una unidad de procesamiento gráfico es la predicción de pKa de la sección 3, que emplea una red neuronal de grafos.

El beneficio de la GPU en ese paso resulta moderado, porque la inferencia se realiza sitio por sitio y el costo de transferencia de datos compensa parcialmente la ganancia. Si se dispone de una T4 sin costo adicional, puede seleccionarse, el código detecta su presencia de forma automática. Si la protonación se mantiene desactivada, la GPU no aporta ningún beneficio.

La configuración recomendada es acceder a `Entorno de ejecución`, luego `Cambiar tipo de entorno`, y seleccionar `T4 GPU` si se planea activar la protonación o `CPU` en caso contrario. Con Colab Pro es aconsejable activar el entorno de ejecución de alta memoria para bibliotecas de miles de compuestos. El resto del código distribuye el trabajo entre los núcleos disponibles y reserva uno para los procesos del sistema.

### 3. Tiempos estimados (basado en 100 moléculas con la versión gratuita de Colab)

| Fase | Tiempo estimado |
|---|---|
| Instalación de dependencias | Aproximadamente 1 minuto |
| Curación y aplicación de filtros | Menos de 1 minuto |
| Instalación de MolGpKa (solo si la protonación está activada) | Entre 2 y 4 minutos |
| Predicción de pKa y protonación en cascada | Entre 1 y 3 minutos |
| Generación de geometría tridimensional | Entre 2 y 5 minutos |

---

## Estado de protonación

Por omisión las moléculas se entregan en estado neutro. El cuaderno incorpora un paso opcional de protonación mediante **MolGpKa**, una red neuronal de grafos que predice el pKa de cada sitio ionizable y asigna el estado de carga al pH indicado. Se describe en la sección 3.

El reporte final conserva la carga formal de cada estructura entregada.

---

## Índice de contenidos

* **0. Entorno de trabajo**
  * [0.1 Dependencias](#s01)
  * [0.2 Configuración](#s02)
  * [0.3 Carga de datos](#s03)
* **1. Curación de estructuras**
  * [1.1 Estandarización](#s11)
  * [1.2 Eliminación de duplicados](#s12)
* **2. Filtrado de moléculas**
  * [2.1 Peso molecular y propiedades](#s21)
  * [2.2 Alertas de reactividad](#s22)
* **3. Estado de protonación con MolGpKa (opcional)**
  * [3.1 Qué es MolGpKa](#s31)
  * [3.2 Instalación de MolGpKa](#s32)
  * [3.3 Asignación del estado de carga](#s33)
* **4. Geometría tridimensional**
  * [4.1 Fundamentos teóricos](#s41)
  * [4.2 Ejecución del algoritmo](#s42)
* **5. Resultados y exportación**
  * [5.1 Revisión visual](#s51)
  * [5.2 Empaquetado de archivos](#s52)
  * [5.3 Descarga de datos](#s53)

<a id="s01"></a>
# 0. Entorno de trabajo

## 0.1 Instalación de dependencias

**Bibliotecas a instalar:** Se configurarán RDKit, que aporta tanto los cálculos de química computacional como su propio módulo de estandarización de estructuras (`rdMolStandardize`), sascorer para evaluar la viabilidad sintética de las moléculas, y py3Dmol junto con mols2grid para la visualización de los compuestos.

**Nota:** Es necesario ejecutar esta celda únicamente una vez al iniciar la sesión de trabajo.

In [17]:
#@title Instalación de dependencias base { display-mode: "form" }
import os, sys, subprocess, textwrap

print("Instalando RDKit y utilidades de visualización...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rdkit", "py3Dmol", "mols2grid"], check=False)

# El SA score vive en Contrib/, fuera del paquete de RDKit.
import urllib.request
for url, destino in [
    ("https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py",
     "sascorer.py"),
    ("https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz",
     "fpscores.pkl.gz"),
]:
    if not os.path.exists(destino):
        urllib.request.urlretrieve(url, destino)
        print(f"  descargado: {destino}")

import rdkit
print(f"\nRDKit versión {rdkit.__version__}  —  listo.")

Instalando RDKit y utilidades de visualización...

RDKit versión 2026.09.1  —  listo.


<a id="s02"></a>
## 0.2 Configuración general

Todos los parámetros que controlan el flujo de trabajo se encuentran en esta celda. No es necesario modificar el código en las celdas posteriores.

| Parámetro | Descripción de su función |
|---|---|
| `CURACION_ESTANDAR` | Elimina sales y metales, estandariza grupos funcionales y neutraliza cargas, mediante el módulo `rdMolStandardize` de RDKit. Es recomendable mantenerlo activado a menos que la base de datos ya haya sido procesada. |
| `TAUTOMERO_CANONICO` | Unifica moléculas que presentan isomería estructural dinámica o tautomería. Ayuda a detectar duplicados ocultos, aunque puede seleccionar un tautómero que no sea el biológicamente activo. Se encuentra desactivado por defecto. |
| `ELIMINAR_DUPLICADOS` | Remueve compuestos idénticos después de realizar la curación. |
| `PM_MINIMO` / `PM_MAXIMO` | Establece el rango de peso molecular en Daltons. Moléculas muy pequeñas ofrecen pocas interacciones específicas, mientras que las muy grandes poseen demasiada flexibilidad para un análisis computacional eficiente. |
| `CATALOGOS_ALERTAS` | Define las bases de datos utilizadas para detectar subestructuras problemáticas. Opciones disponibles: `PAINS`, `BRENK`, `NIH`, `ZINC`, `CHEMBL`. |
| `DESCARTAR_ALERTADAS` | Si es falso, el programa etiqueta la molécula pero la conserva, lo cual es la opción sugerida. Si es verdadero, la molécula se elimina del estudio antes del acoplamiento molecular. |
| `CALCULAR_SA_SCORE` | Evalúa qué tan difícil sería sintetizar la molécula en un laboratorio en una escala del 1 al 10. Útil para priorizar compuestos tras el acoplamiento. |
| `N_CONFORMEROS_PRUEBA` | Determina el número de conformaciones espaciales generadas por molécula. Solo se exportará la configuración de menor energía. Un número mayor mejora la geometría resultante pero requiere más tiempo de procesamiento. |
| `SEMILLA_ALEATORIA` | Garantiza que los resultados sean reproducibles al ejecutar el código nuevamente. |
| `MAX_ITER_MMFF` | Define el número máximo de iteraciones permitidas durante la minimización de la energía molecular. |
| `N_NUCLEOS` | Asigna de manera automática todos los núcleos de procesamiento disponibles exceptuando uno para mantener la estabilidad del sistema. |

In [18]:
#@title Configuración del flujo { display-mode: "form" }
import multiprocessing

#@markdown ### 1 · Curación
CURACION_ESTANDAR = True #@param {type:"boolean"}
TAUTOMERO_CANONICO = False #@param {type:"boolean"}
ELIMINAR_DUPLICADOS = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 2 · Filtrado por peso molecular
FILTRO_PESO_MOLECULAR = True #@param {type:"boolean"}
PM_MINIMO = 150 #@param {type:"slider", min:0, max:600, step:10}
PM_MAXIMO = 800 #@param {type:"slider", min:200, max:2000, step:50}

#@markdown ---
#@markdown ### 3 · Alertas estructurales
FILTRO_TOXICOLOGICO = True #@param {type:"boolean"}
#@markdown Catálogos a aplicar:
PAINS = True #@param {type:"boolean"}
BRENK = True #@param {type:"boolean"}
NIH = True #@param {type:"boolean"}
ZINC = False #@param {type:"boolean"}
CHEMBL = False #@param {type:"boolean"}
ACCION_ALERTADAS = "Etiquetar y conservar" #@param ["Etiquetar y conservar", "Descartar"]
CALCULAR_SA_SCORE = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 4 · Estado de protonación (opcional)
PROTONAR = True #@param {type:"boolean"}
PH_OBJETIVO = 7.4 #@param {type:"slider", min:1.0, max:14.0, step:0.1}

#@markdown ---
#@markdown ### 5 · Geometría tridimensional
N_CONFORMEROS_PRUEBA = 10 #@param {type:"slider", min:1, max:50, step:1}

#@markdown ---
#@markdown ### 6 · Archivos de salida
NOMBRE_SALIDA = "Prueba_1" #@param {type:"string"}
GUARDAR_SDF_INDIVIDUALES = True #@param {type:"boolean"}
GUARDAR_SDF_UNICO = True #@param {type:"boolean"}

# ── Parámetros fijos ────────────────────────────────────────────────────
MARGEN_MINIMO      = 1.0   # unidades de pKa; ver sección 3.3
DISTANCIA_AVISO    = 4     # enlaces; solo define el umbral del aviso
MAX_RONDAS_CASCADA = 6     # tope de ionizaciones sucesivas
USAR_GPU_SI_EXISTE = True  # se detecta de forma automática
MAX_ITER_MMFF      = 2000  # iteraciones de la minimización MMFF94
SEMILLA_ALEATORIA  = 99    # reproducibilidad del embedding

# ── Parámetros derivados ────────────────────────────────────────────────
CATALOGOS_ALERTAS = [n for n, activo in
                     (("PAINS", PAINS), ("BRENK", BRENK), ("NIH", NIH),
                      ("ZINC", ZINC), ("CHEMBL", CHEMBL)) if activo]
DESCARTAR_ALERTADAS = (ACCION_ALERTADAS == "Descartar")
N_NUCLEOS = max(1, multiprocessing.cpu_count() - 1)

PM_MINIMO, PM_MAXIMO = float(PM_MINIMO), float(PM_MAXIMO)
if PM_MINIMO >= PM_MAXIMO:
    raise ValueError(f"PM_MINIMO ({PM_MINIMO}) debe ser menor que PM_MAXIMO ({PM_MAXIMO}).")

print(f"Núcleos            : {N_NUCLEOS} de {multiprocessing.cpu_count()}")
print(f"Peso molecular     : {PM_MINIMO:.0f} a {PM_MAXIMO:.0f} Da"
      if FILTRO_PESO_MOLECULAR else "Peso molecular     : sin filtro")
print(f"Alertas            : {', '.join(CATALOGOS_ALERTAS) or 'ninguna'}"
      f" ({'descartar' if DESCARTAR_ALERTADAS else 'etiquetar'})")
if PROTONAR:
    print(f"Protonación        : pH {PH_OBJETIVO}, margen {MARGEN_MINIMO}")
else:
    print("Protonación        : desactivada (estructuras neutras)")

Núcleos            : 1 de 2
Peso molecular     : 150 a 800 Da
Alertas            : PAINS, BRENK, NIH (etiquetar)
Protonación        : pH 7.4, margen 1.0


<a id="s03"></a>
## 0.3 Cargar el CSV

**Descripción del proceso:** se realiza la lectura del archivo, se normalizan los nombres de las dos columnas principales, se eliminan espacios en blanco y se renombran los identificadores duplicados para mantener la trazabilidad.

Al ejecutar la celda, se desplegará un botón para cargar el archivo. En un entorno ajeno a Colab, se debe utilizar la función `cargar_csv("ruta/archivo.csv")`.

In [19]:
#@title Subir el archivo CSV { display-mode: "form" }
import pandas as pd, os, io

def cargar_csv(ruta=None):
    """Carga el CSV y normaliza ID y SMILES."""
    if ruta is None:
        try:
            from google.colab import files
            subidos = files.upload()
            nombre = list(subidos.keys())[0]
            df = pd.read_csv(io.BytesIO(subidos[nombre]))
            print(f"Archivo cargado: {nombre}")
        except ImportError:
            raise RuntimeError("Fuera de Colab: indica la ruta con cargar_csv('archivo.csv')")
    else:
        df = pd.read_csv(ruta)

    # Encabezados sin espacios y en minúsculas.
    mapa = {}
    for col in df.columns:
        limpio = str(col).strip().lower()
        if limpio == "id":
            mapa[col] = "ID"
        elif limpio in ("smiles", "smile", "canonical_smiles"):
            mapa[col] = "SMILES"
    df = df.rename(columns=mapa)

    faltantes = [c for c in ("ID", "SMILES") if c not in df.columns]
    if faltantes:
        raise KeyError(
            f"Faltan las columnas {faltantes}. "
            f"El archivo tiene: {list(df.columns)}"
        )

    df = df[["ID", "SMILES"]].copy()
    df["ID"] = df["ID"].astype(str).str.strip()
    df["SMILES"] = df["SMILES"].astype(str).str.strip()

    # ID repetido: se le añade sufijo.
    if df["ID"].duplicated().any():
        n_rep = int(df["ID"].duplicated().sum())
        df["ID"] = df["ID"] + df.groupby("ID").cumcount().apply(
            lambda i: "" if i == 0 else f"_{i+1}")
        print(f"Aviso: había {n_rep} identificadores repetidos; se renombraron.")

    return df.reset_index(drop=True)


df_entrada = cargar_csv()
print(f"\nMoléculas de entrada: {len(df_entrada)}")
display(df_entrada.head())

Saving test_100_moleculas (1).csv to test_100_moleculas (1) (2).csv
Archivo cargado: test_100_moleculas (1) (2).csv

Moléculas de entrada: 100


,ID,SMILES
0,448219,[H][C@](N)(CCCC)C(O)(O)CNC1CC1
1,49866822,NC[C@@]1(O)NC(=O)C(O)=[N+]1CC(O)=O
2,93485622,[H]N([H])[C@@H](C[C@@H](F)CC\N=C(/C)N([H])[H])...
3,24764436,N1C=CC(=N1)C1=CC(=NO1)C1=CC=CC=C1
4,4909,CCC1(C(=O)NCNC1=O)C1=CC=CC=C1


<a id="s11"></a>
# 1. Curación

## 1.1 Estandarización

**Procedimiento de la celda:** se sanea la estructura → se eliminan hidrógenos explícitos → se desconectan metales → se conserva el fragmento de mayor tamaño → se normalizan los grupos funcionales → se reioniza → se neutraliza → se reasigna la estereoquímica.

**Consideraciones adicionales:**

- Los metales se detectan analizando el símbolo atómico de cada elemento.
- Si un centro quiral carece de configuración definida, se le asigna una orientación estérica estática y reproducible. Dicha modificación queda registrada en la columna `estereo_indefinida` para su posterior revisión.

In [20]:
#@title Función de curación { display-mode: "form" }
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from multiprocessing import Pool
import pandas as pd

RDLogger.DisableLog("rdApp.*")   # silencia los avisos por molécula

# Elementos esperados en química medicinal.
ELEMENTOS_PERMITIDOS = {"C","H","O","N","P","S","F","Cl","Br","I","B","Si","Se"}

# Metales: hacen fallar el campo de fuerza.
METALES = {
    "Li","Na","K","Rb","Cs","Be","Mg","Ca","Sr","Ba","Al","Ga","In","Tl",
    "Sn","Pb","Sb","Bi","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Zr",
    "Mo","Ru","Rh","Pd","Ag","Cd","W","Re","Os","Ir","Pt","Au","Hg","Gd",
}


def _curar_una(args):
    """Estandariza un SMILES. Devuelve el resultado o el motivo del descarte.
    Solo tipos simples: se ejecuta en paralelo."""
    mol_id, smiles, estandarizar, hacer_tautomero = args

    salida = {"ID": mol_id, "SMILES_original": smiles,
              "SMILES_curado": None, "elementos_no_usuales": False,
              "contiene_metal": False, "estereo_indefinida": False,
              "estado": "ok", "motivo": ""}

    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            salida["estado"] = "descartada"
            salida["motivo"] = "SMILES ilegible o valencia imposible"
            return salida

        mol = Chem.RemoveHs(mol)

        if estandarizar:
            mol = rdMolStandardize.MetalDisconnector().Disconnect(mol)
            mol = rdMolStandardize.FragmentParent(mol)       # fragmento mayor
            mol = rdMolStandardize.Normalize(mol)
            mol = rdMolStandardize.Reionize(mol)
            mol = rdMolStandardize.Uncharger().uncharge(mol)

        if hacer_tautomero:
            mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)

        Chem.AssignStereochemistry(mol, force=True, cleanIt=True)

        # Por símbolo atómico, no por texto del SMILES.
        simbolos = {a.GetSymbol() for a in mol.GetAtoms()}
        salida["contiene_metal"] = len(simbolos & METALES) > 0
        salida["elementos_no_usuales"] = len(simbolos - ELEMENTOS_PERMITIDOS) > 0

        if salida["contiene_metal"]:
            salida["estado"] = "descartada"
            salida["motivo"] = ("contiene metal ("
                                + ", ".join(sorted(simbolos & METALES)) + ")")
            return salida

        # Centros quirales sin especificar.
        sin_asignar = Chem.FindMolChiralCenters(
            mol, includeUnassigned=True, useLegacyImplementation=False)
        salida["estereo_indefinida"] = any(c[1] == "?" for c in sin_asignar)

        salida["SMILES_curado"] = Chem.MolToSmiles(mol, isomericSmiles=True)
        return salida

    except Exception as e:
        salida["estado"] = "descartada"
        salida["motivo"] = f"error de curación: {type(e).__name__}"
        return salida


def curar_biblioteca(df, n_nucleos, estandarizar=True, hacer_tautomero=False):
    tareas = [(r.ID, r.SMILES, estandarizar, hacer_tautomero)
              for r in df.itertuples(index=False)]
    if n_nucleos > 1:
        with Pool(n_nucleos) as pool:
            resultados = pool.map(_curar_una, tareas, chunksize=50)
    else:
        resultados = [_curar_una(t) for t in tareas]
    return pd.DataFrame(resultados)


print("Función de curación definida.")

Función de curación definida.


**Ejecutar la curación.** Con `TAUTOMERO_CANONICO = True` este paso es bastante más
lento.

In [21]:
#@title Ejecutar la curación { display-mode: "form" }
import time

t0 = time.time()
df_curado = curar_biblioteca(
    df_entrada,
    n_nucleos=N_NUCLEOS,
    estandarizar=CURACION_ESTANDAR,
    hacer_tautomero=TAUTOMERO_CANONICO,
)
t1 = time.time()

n_ok   = int((df_curado["estado"] == "ok").sum())
n_fail = len(df_curado) - n_ok

print(f"Tiempo: {t1 - t0:.1f} s")
print(f"Curadas correctamente : {n_ok}")
print(f"Descartadas           : {n_fail}")

if n_fail:
    print("\nMotivos de descarte:")
    print(df_curado.loc[df_curado["estado"] != "ok", "motivo"]
          .value_counts().to_string())

n_estereo = int(df_curado["estereo_indefinida"].sum())
n_raros   = int(df_curado["elementos_no_usuales"].sum())
if n_estereo:
    print(f"\n{n_estereo} moléculas tienen al menos un centro quiral sin definir "
          f"(se les asignará una configuración fija y reproducible).")
if n_raros:
    print(f"{n_raros} moléculas contienen elementos fuera de la lista habitual "
          f"(se conservan, quedan etiquetadas).")

display(df_curado.head())

Tiempo: 0.6 s
Curadas correctamente : 100
Descartadas           : 0

13 moléculas tienen al menos un centro quiral sin definir (se les asignará una configuración fija y reproducible).


,ID,SMILES_original,SMILES_curado,elementos_no_usuales,contiene_metal,estereo_indefinida,estado,motivo
0,448219,[H][C@](N)(CCCC)C(O)(O)CNC1CC1,CCCC[C@H](N)C(O)(O)CNC1CC1,False,False,False,ok,
1,49866822,NC[C@@]1(O)NC(=O)C(O)=[N+]1CC(O)=O,NC[C@@]1(O)NC(=O)C(=O)N1CC(=O)O,False,False,False,ok,
2,93485622,[H]N([H])[C@@H](C[C@@H](F)CC\N=C(/C)N([H])[H])...,C/C(N)=N\CC[C@H](F)C[C@H](N)C(=O)O,False,False,False,ok,
3,24764436,N1C=CC(=N1)C1=CC(=NO1)C1=CC=CC=C1,c1ccc(-c2cc(-c3cc[nH]n3)on2)cc1,False,False,False,ok,
4,4909,CCC1(C(=O)NCNC1=O)C1=CC=CC=C1,CCC1(c2ccccc2)C(=O)NCNC1=O,False,False,False,ok,


<a id="s12"></a>
## 1.2 Duplicados

**Fundamento teórico:** una vez concluida la fase de curación, todas las cadenas SMILES adoptan una forma canónica, por lo que dos moléculas idénticas producirán de forma estricta la misma representación textual. La comparación de estas cadenas resulta suficiente para la identificación y eliminación de compuestos repetidos.

In [22]:
#@title Detectar y eliminar duplicados { display-mode: "form" }

df_validas = df_curado[df_curado["estado"] == "ok"].copy()

df_duplicados = pd.DataFrame(columns=["SMILES_curado", "ID_conservado", "ID_eliminado"])

if ELIMINAR_DUPLICADOS and len(df_validas):
    marcados = df_validas[df_validas.duplicated("SMILES_curado", keep=False)]
    filas = []
    for smi, grupo in marcados.groupby("SMILES_curado"):
        ids = list(grupo["ID"])
        for eliminado in ids[1:]:
            filas.append({"SMILES_curado": smi,
                          "ID_conservado": ids[0],
                          "ID_eliminado": eliminado})
    df_duplicados = pd.DataFrame(filas)

    antes = len(df_validas)
    df_validas = df_validas.drop_duplicates("SMILES_curado", keep="first")
    print(f"Duplicados eliminados: {antes - len(df_validas)}")
    if len(df_duplicados):
        display(df_duplicados.head(10))
else:
    print("Eliminación de duplicados desactivada.")

print(f"\nMoléculas que continúan: {len(df_validas)}")

Duplicados eliminados: 0

Moléculas que continúan: 100


<a id="s21"></a>
# 2. Filtros y descriptores

## 2.1 Peso molecular y descriptores

**Fundamento del filtro:** permite acotar el espacio químico a evaluar estableciendo límites físicos y termodinámicos.

- *Límite inferior:* en moléculas que poseen escasos átomos pesados existen pocas áreas de contacto para facilitar interacciones de alta especificidad. En consecuencia, el algoritmo de acoplamiento molecular tiende a sobrestimar la afinidad relativa por átomo.
- *Límite superior:* el espacio conformacional experimenta un crecimiento cuasiexponencial en función del número de enlaces rotables. Una alta flexibilidad estructural excede la capacidad de muestreo de los métodos de acoplamiento molecular rígido o semirígido.

Se emplea la **masa exacta monoisotópica**, la cual garantiza alta reproducibilidad numérica.

**Descripción del procedimiento:** se procede al cálculo del peso molecular, logP, área de superficie polar topológica (TPSA), donadores y aceptores de enlaces de hidrógeno, enlaces rotables, anillos aromáticos, índice QED y accesibilidad sintética (SA score). Posteriormente, se aplica el filtro restrictivo de peso molecular; el resto de los descriptores se incorpora al reporte final para el análisis correspondiente.

**Accesibilidad sintética (SA score):** esta métrica opera en una escala del 1 al 10. Se sugiere emplear este descriptor en el análisis retrospectivo; frente a energías de acoplamiento equivalentes, se debe priorizar a los compuestos con mayor facilidad de síntesis.

In [23]:
#@title Calcular peso molecular y descriptores { display-mode: "form" }
from rdkit.Chem import Descriptors, rdMolDescriptors, QED

def _descriptores_una(args):
    """Descriptores de una molécula."""
    mol_id, smiles, calcular_sa = args
    d = {"ID": mol_id, "PM": None, "enlaces_rotables": None,
         "anillos_aromaticos": None, "donadores_H": None, "aceptores_H": None,
         "logP": None, "TPSA": None, "QED": None, "SA_score": None}
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return d
    d["PM"]                 = round(Descriptors.ExactMolWt(mol), 2)
    d["enlaces_rotables"]   = rdMolDescriptors.CalcNumRotatableBonds(mol)
    d["anillos_aromaticos"] = rdMolDescriptors.CalcNumAromaticRings(mol)
    d["donadores_H"]        = rdMolDescriptors.CalcNumHBD(mol)
    d["aceptores_H"]        = rdMolDescriptors.CalcNumHBA(mol)
    d["logP"]               = round(Descriptors.MolLogP(mol), 2)
    d["TPSA"]               = round(rdMolDescriptors.CalcTPSA(mol), 2)
    try:
        d["QED"] = round(QED.qed(mol), 3)
    except Exception:
        pass
    if calcular_sa:
        try:
            import sascorer
            d["SA_score"] = round(sascorer.calculateScore(mol), 2)
        except Exception:
            pass
    return d


tareas = [(r.ID, r.SMILES_curado, CALCULAR_SA_SCORE)
          for r in df_validas.itertuples(index=False)]

if N_NUCLEOS > 1:
    with Pool(N_NUCLEOS) as pool:
        desc = pool.map(_descriptores_una, tareas, chunksize=50)
else:
    desc = [_descriptores_una(t) for t in tareas]

df_validas = df_validas.merge(pd.DataFrame(desc), on="ID", how="left")

# Filtro de peso molecular.
if FILTRO_PESO_MOLECULAR:
    dentro = df_validas["PM"].between(PM_MINIMO, PM_MAXIMO)
    fuera  = df_validas.loc[~dentro].copy()
    fuera["motivo"] = "peso molecular fuera de rango"
    print(f"Fuera del rango {PM_MINIMO}–{PM_MAXIMO} Da: {len(fuera)}")
    df_validas = df_validas.loc[dentro].copy()
else:
    fuera = pd.DataFrame()
    print("Filtro de peso molecular desactivado.")

print(f"Moléculas que continúan: {len(df_validas)}")
display(df_validas[["ID","PM","logP","TPSA","enlaces_rotables","QED","SA_score"]].head())

Fuera del rango 150.0–800.0 Da: 8
Moléculas que continúan: 92


,ID,PM,logP,TPSA,enlaces_rotables,QED,SA_score
0,448219,202.17,-0.06,78.51,7,0.436,3.40
1,49866822,203.05,-3.37,132.96,3,0.354,3.89
2,93485622,205.12,-0.11,101.70,6,0.416,3.88
3,24764436,211.07,2.73,54.71,2,0.709,2.60
4,4909,218.11,0.54,58.20,2,0.712,2.77


<a id="s22"></a>
## 2.2 Alertas estructurales

**Definición:** consiste en el rastreo y la identificación de subestructuras asociadas con comportamientos farmacocinéticos o toxicológicos desfavorables. Cabe destacar que este proceso no constituye un modelo predictivo de toxicidad clínica, sino una advertencia estructural temprana.

| Catálogo | Identificación principal |
|---|---|
| **PAINS** | Compuestos que generan falsos positivos en ensayos biológicos por mecanismos inespecíficos: agregación coloidal, interferencia por fluorescencia, etc. |
| **BRENK** | Presencia de grupos funcionales indeseables en las etapas tempranas del diseño de fármacos, tales como nitro aromáticos y aceptores de Michael. |
| **NIH** | Estructuras que exhiben reactividad o interferencia empíricamente demostrada en ensayos del NIH. |
| **ZINC** / **CHEMBL** | Conjuntos de reglas heurísticas más permisivas derivadas de los criterios de inclusión de estas bases de datos. |

**Consideración analítica:** existen fármacos aprobados para uso clínico que desencadenan este tipo de alertas. Por ejemplo, la eficacia del metronidazol recae en la bioactivación de su grupo nitro. Por este motivo, se sugiere emplear `DESCARTAR_ALERTADAS = False`, permitiendo realizar la depuración correspondiente una vez calculadas las energías de interacción.

**Descripción del proceso:** la celda reporta todas las alertas detectadas para cada ligando en las columnas correspondientes del reporte, brindando la posibilidad de conservar o excluir dichas moléculas del análisis según la configuración elegida.

In [24]:
#@title Filtro de alertas estructurales { display-mode: "form" }
from rdkit.Chem import FilterCatalog

def construir_catalogo(nombres):
    """Catálogo combinado de alertas."""
    params = FilterCatalog.FilterCatalogParams()
    disponibles = {
        "PAINS":  FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS,
        "BRENK":  FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK,
        "NIH":    FilterCatalog.FilterCatalogParams.FilterCatalogs.NIH,
        "ZINC":   FilterCatalog.FilterCatalogParams.FilterCatalogs.ZINC,
        "CHEMBL": FilterCatalog.FilterCatalogParams.FilterCatalogs.CHEMBL,
    }
    for n in nombres:
        clave = n.strip().upper()
        if clave in disponibles:
            params.AddCatalog(disponibles[clave])
        else:
            print(f"  aviso: catálogo '{n}' no reconocido, se omite")
    return FilterCatalog.FilterCatalog(params)


_catalogo = construir_catalogo(CATALOGOS_ALERTAS) if FILTRO_TOXICOLOGICO else None


def _alertas_una(args):
    """Todas las alertas de una molécula."""
    mol_id, smiles = args
    r = {"ID": mol_id, "alerta_tox": "no evaluado",
         "alerta_detalle": "", "n_alertas": 0}
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return r
    coincidencias = _catalogo.GetMatches(mol)
    if coincidencias:
        nombres = sorted({m.GetDescription() for m in coincidencias})
        r["alerta_tox"]     = "alerta"
        r["alerta_detalle"] = "; ".join(nombres)
        r["n_alertas"]      = len(nombres)
    else:
        r["alerta_tox"] = "segura"
    return r


if FILTRO_TOXICOLOGICO and len(df_validas):
    tareas = [(r.ID, r.SMILES_curado) for r in df_validas.itertuples(index=False)]
    # En serie: el catálogo es un objeto de C++, costoso de copiar.
    resultados = [_alertas_una(t) for t in tareas]
    df_validas = df_validas.merge(pd.DataFrame(resultados), on="ID", how="left")

    n_alert = int((df_validas["alerta_tox"] == "alerta").sum())
    print(f"Catálogos usados : {', '.join(CATALOGOS_ALERTAS)}")
    print(f"Con al menos una alerta : {n_alert} de {len(df_validas)}")

    if n_alert:
        conteo = (df_validas.loc[df_validas["alerta_tox"] == "alerta",
                                 "alerta_detalle"]
                  .str.split("; ").explode().value_counts().head(12))
        print("\nAlertas más frecuentes:")
        print(conteo.to_string())

    if DESCARTAR_ALERTADAS:
        df_validas = df_validas[df_validas["alerta_tox"] != "alerta"].copy()
        print(f"\nDescartadas por alerta. Continúan: {len(df_validas)}")
    else:
        print("\nLas moléculas alertadas se conservan y quedan etiquetadas "
              "en el reporte final.")
else:
    df_validas["alerta_tox"]     = "no evaluado"
    df_validas["alerta_detalle"] = ""
    df_validas["n_alertas"]      = 0
    print("Filtro toxicológico desactivado.")

Catálogos usados : PAINS, BRENK, NIH
Con al menos una alerta : 50 de 92

Alertas más frecuentes:
alerta_detalle
Oxygen-nitrogen_single_bond    12
nitro_group                    10
imine_1                         8
Aliphatic_long_chain            7
imine_2                         6
phosphor                        5
quaternary_nitrogen_1           4
Michael_acceptor_1              4
catechol_A(92)                  3
isolated_alkene                 3
aniline                         3
catechol                        3

Las moléculas alertadas se conservan y quedan etiquetadas en el reporte final.


**Resumen estadístico de los descriptores calculados.**

In [25]:
#@title Resumen de los descriptores calculados { display-mode: "form" }
if len(df_validas):
    columnas = ["PM", "logP", "TPSA", "donadores_H", "aceptores_H",
                "enlaces_rotables", "anillos_aromaticos", "QED", "SA_score"]
    columnas = [c for c in columnas if c in df_validas.columns]
    display(df_validas[columnas].describe().T.round(2))
else:
    print("No quedan moléculas para describir. Revisa los filtros aplicados.")

,count,mean,std,min,25%,50%,75%,max
PM,92.0,368.59,111.02,202.17,288.15,350.65,435.33,688.26
logP,92.0,2.37,2.14,-3.48,1.03,2.56,3.86,8.05
TPSA,92.0,94.82,47.42,3.24,66.93,88.06,112.88,299.48
donadores_H,92.0,2.21,1.64,0.00,1.00,2.00,3.00,7.00
aceptores_H,92.0,5.28,2.48,2.00,4.00,5.00,7.00,16.00
enlaces_rotables,92.0,5.03,2.76,0.00,3.00,5.00,6.00,14.00
anillos_aromaticos,92.0,1.91,1.24,0.00,1.00,2.00,3.00,7.00
QED,92.0,0.56,0.21,0.05,0.42,0.59,0.73,0.91
SA_score,92.0,3.13,0.81,1.54,2.53,2.91,3.68,5.49


<a id="s31"></a>
# 3. Estado de protonación (opcional)

## 3.1 Qué es MolGpKa

Este paso permanece desactivado por omisión.

Se utilizó **MolGpKa**, un modelo de aprendizaje profundo para la predicción de pKa de
moléculas pequeñas a partir de una red neuronal de gráfos.

### Relación entre pKa y pH (Ecuación de Henderson-Hasselbalch)

Para un ácido débil que se disocia según la reacción $\text{HA} \rightleftharpoons \text{H}^+ + \text{A}^-$, la relación entre las especies se define como:

$$ \frac{[\text{A}^-]}{[\text{HA}]} = 10^{(\text{pH} - \text{pKa})} $$

Donde:
* **pH:** es el logaritmo decimal negativo de la concentración de protones.
* **pKa:** es el pH al que la mitad de las moléculas conservan el protón en el sitio considerado.
* **[A⁻]:** es la concentración de la forma desprotonada (ionizada).
* **[HA]:** es la concentración de la forma protonada (neutra).

### Cálculo del porcentaje de sustancia ionizada

La ecuación indica que por cada unidad en que el pH supera al pKa, la forma desprotonada resulta diez veces más abundante.

**Ejemplo de cálculo:**
Para un sitio ionizable con un **pKa de 4.2** a un **pH fisiológico de 7.4**:

1. Se calcula la diferencia: $\text{pH} - \text{pKa} = 7.4 - 4.2 = 3.2$
2. Se obtiene la razón: $10^{3.2} \approx 1585$
3. Esto significa que hay 1585 moléculas en forma desprotonada por cada molécula en forma protonada.
4. Convertido a porcentaje, esto es equivalente al **99.94 % de la sustancia en forma ionizada**.

In [26]:
#@title Instalar MolGpKa y aplicar correcciones { display-mode: "form" }
import os, sys, subprocess

MOLGPKA_LISTO = False
RUTA_MOLGPKA  = os.path.abspath("MolGpKa")
DISPOSITIVO   = "cpu"


def _correr(cmd, descripcion, critico=True):
    """Ejecuta un comando; si no es crítico, continúa tras el fallo."""
    print(f"  {descripcion}...")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        aviso = "FALLÓ" if critico else "no disponible (se continúa)"
        print(f"    {aviso}: {(r.stderr or '').strip()[-300:]}")
        return not critico
    return True


def _parchar_compatibilidad(raiz):
    """Las tres correcciones de la sección 3.2."""
    cambios = []

    # 1. Ruta del archivo de patrones.
    f_ion = os.path.join(raiz, "src", "utils", "ionization_group.py")
    with open(f_ion) as f:
        txt = f.read()
    malo  = 'root = os.path.join(os.path.abspath(""), "utils")'
    bueno = 'root = os.path.abspath(os.path.dirname(__file__))'
    if malo in txt:
        txt = txt.replace(malo, bueno)
        with open(f_ion, "w") as f:
            f.write(txt)
        cambios.append("ruta de smarts_pattern.tsv")

    # 2 y 3. Importaciones de PyTorch Geometric.
    f_net = os.path.join(raiz, "src", "utils", "net.py")
    with open(f_net) as f:
        txt = f.read()

    muerta = "from torch_geometric.data import DataLoader\n"
    if muerta in txt:
        txt = txt.replace(muerta, "")
        cambios.append("importación obsoleta de DataLoader")

    vieja = "from torch_geometric.nn import GATConv, GlobalAttention\n"
    nueva = (
        "from torch_geometric.nn import GATConv\n"
        "try:  # PyG reciente\n"
        "    from torch_geometric.nn.aggr import AttentionalAggregation "
        "as GlobalAttention\n"
        "except ImportError:  # PyG anterior\n"
        "    from torch_geometric.nn import GlobalAttention\n"
    )
    if vieja in txt:
        txt = txt.replace(vieja, nueva)
        cambios.append("reubicación de GlobalAttention")

    with open(f_net, "w") as f:
        f.write(txt)

    return cambios


if not PROTONAR:
    print("Protonación desactivada: no se instala MolGpKa.")
else:
    print("Preparando MolGpKa\n")
    ok = True

    if not os.path.isdir(RUTA_MOLGPKA):
        ok = _correr(
            "git clone --depth 1 -q https://github.com/Xundrug/MolGpKa.git MolGpKa",
            "clonando el repositorio")
    else:
        print("  repositorio ya presente")

    if ok:
        try:
            import torch_geometric  # noqa: F401
            print("  PyTorch Geometric ya disponible")
        except ImportError:
            ok = _correr(f"{sys.executable} -m pip install -q torch_geometric",
                         "instalando PyTorch Geometric")

    # Extensión compilada: opcional, no interrumpe si falla.
    if ok:
        try:
            import torch_scatter  # noqa: F401
            print("  torch-scatter ya disponible")
        except ImportError:
            import torch
            indice = f"https://data.pyg.org/whl/torch-{torch.__version__}.html"
            _correr(f"{sys.executable} -m pip install -q torch-scatter -f {indice}",
                    "instalando torch-scatter", critico=False)

    if ok:
        cambios = _parchar_compatibilidad(RUTA_MOLGPKA)
        if cambios:
            print("  correcciones aplicadas: " + ", ".join(cambios))
        else:
            print("  sin correcciones pendientes")

    if ok:
        # Importaciones planas: src debe ir en sys.path.
        ruta_src = os.path.join(RUTA_MOLGPKA, "src")
        if ruta_src not in sys.path:
            sys.path.insert(0, ruta_src)
        try:
            import torch
            from predict_pka import load_model, model_pred
            from utils.ionization_group import get_ionization_aid
            MOLGPKA_LISTO = True
            if USAR_GPU_SI_EXISTE and torch.cuda.is_available():
                DISPOSITIVO = "cuda"
            torch.set_num_threads(N_NUCLEOS)
            print(f"\nMolGpKa disponible. Dispositivo: {DISPOSITIVO}")
            if DISPOSITIVO == "cuda":
                print(f"  GPU: {torch.cuda.get_device_name(0)}")
        except Exception as e:
            print(f"\nNo fue posible importar MolGpKa: {type(e).__name__}: {e}")
            print("El cuaderno continuará con las estructuras neutras.")
    else:
        print("\nLa preparación falló. Se continuará con las estructuras neutras.")

Preparando MolGpKa

  repositorio ya presente
  PyTorch Geometric ya disponible
  torch-scatter ya disponible
  sin correcciones pendientes

MolGpKa disponible. Dispositivo: cuda
  GPU: Tesla T4


In [27]:
#@title Predecir pKa y protonar en cascada { display-mode: "form" }
import time
from rdkit.Chem.MolStandardize import rdMolStandardize

# Respaldo: si la configuración en memoria es de otra versión, se completa.
if "PROTONAR" not in dir():
    raise NameError(
        "Falta la celda de configuración: ejecuta primero la sección 0.2.")

_POR_OMISION = {"PH_OBJETIVO": 7.4, "MARGEN_MINIMO": 1.0,
                "DISTANCIA_AVISO": 4, "MAX_RONDAS_CASCADA": 6}
_faltantes = []
for _nombre, _valor in _POR_OMISION.items():
    if _nombre not in dir():
        globals()[_nombre] = _valor
        _faltantes.append(f"{_nombre}={_valor}")
if _faltantes:
    print("Aviso: estos parámetros no estaban definidos y se usan por omisión: "
          + ", ".join(_faltantes))
    print("Conviene volver a ejecutar la celda de configuración (sección 0.2).\n")

# Por omisión, estructura neutra.
df_validas["SMILES_final"]        = df_validas["SMILES_curado"]
df_validas["protonacion"]         = "desactivada"
df_validas["pka_acidos"]          = ""
df_validas["pka_basicos"]         = ""
df_validas["n_sitios_ionizables"] = 0
df_validas["ionizaciones_proximas"] = 0


def _modificar_acido(atomo):
    """Desprotona: carga −1, un hidrógeno menos.
    GetTotalNumHs y no GetNumExplicitHs, que da cero si son implícitos."""
    atomo.SetFormalCharge(-1)
    atomo.SetNumExplicitHs(max(0, atomo.GetTotalNumHs() - 1))
    atomo.SetNoImplicit(True)


def _modificar_base(atomo):
    """Protona: carga +1, un hidrógeno más."""
    atomo.SetFormalCharge(1)
    atomo.SetNumExplicitHs(atomo.GetTotalNumHs() + 1)
    atomo.SetNoImplicit(True)


def _neutralizar(smiles):
    """Punto de partida neutro."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = rdMolStandardize.Uncharger().uncharge(mol)
    return Chem.MolFromSmiles(Chem.MolToSmiles(mol))


def _sitios_con_pka(molh, n_pesados, modelo_acido, modelo_base, dispositivo):
    """pKa de cada sitio, como (pka, índice del átomo pesado, tipo).
    Los sitios ácidos llegan como índice de hidrógeno y se traducen."""
    sitios = []
    for tipo, modelo, etiqueta in (("A", modelo_acido, "acid"),
                                   ("B", modelo_base,  "base")):
        for idx in get_ionization_aid(molh, acid_or_base=etiqueta):
            pka = float(model_pred(molh, idx, modelo, dispositivo))
            if tipo == "A":
                vecinos = molh.GetAtomWithIdx(idx).GetNeighbors()
                if not vecinos:
                    continue
                pesado = vecinos[0].GetIdx()
            else:
                pesado = idx
            # AddHs añade al final: los pesados conservan su índice.
            if pesado < n_pesados:
                sitios.append((pka, pesado, tipo))
    return sitios


def protonar_cascada(smiles_neutro, modelo_acido, modelo_base, dispositivo,
                     ph, margen_min, distancia_aviso, max_rondas):
    """Ioniza un sitio por ronda, re-prediciendo sobre el estado cargado.
    Sin corrección del pKa; ver sección 3.3."""
    base = _neutralizar(smiles_neutro)
    if base is None:
        return None, [], [], [], 0, "estructura ilegible"

    mol        = Chem.RWMol(base)
    n_pesados  = mol.GetNumAtoms()
    distancias = Chem.GetDistanceMatrix(base)   # la conectividad no cambia
    ionizados  = {"A": set(), "B": set()}
    pka_ini    = {"A": [], "B": []}
    bitacora   = []

    for ronda in range(max_rondas):
        actual = mol.GetMol()
        try:
            Chem.SanitizeMol(actual)
        except Exception:
            return (None, pka_ini["A"], pka_ini["B"], bitacora, 0,
                    "sanitización fallida")

        try:
            sitios = _sitios_con_pka(Chem.AddHs(actual), n_pesados,
                                     modelo_acido, modelo_base, dispositivo)
        except Exception as e:
            return (None, pka_ini["A"], pka_ini["B"], bitacora, 0,
                    f"error de predicción: {type(e).__name__}")

        if ronda == 0:
            for pka, _pesado, tipo in sitios:
                pka_ini[tipo].append(pka)

        viables = []
        for pka, pesado, tipo in sitios:
            if pesado in ionizados[tipo]:
                continue
            # pKa tal como lo predice el modelo.
            margen = (ph - pka) if tipo == "A" else (pka - ph)
            if margen <= margen_min:
                continue
            viables.append((margen, pka, pesado, tipo))

        if not viables:
            break

        viables.sort(key=lambda t: -t[0])
        _margen, pka, pesado, tipo = viables[0]
        atomo = mol.GetAtomWithIdx(pesado)
        if tipo == "A":
            _modificar_acido(atomo)
        else:
            _modificar_base(atomo)
        ionizados[tipo].add(pesado)
        bitacora.append(f"{tipo}{pesado}:{pka:.2f}")

    # Diagnóstico, no corrección: pares del mismo signo que quedaron próximos.
    proximas = 0
    for tipo in ("A", "B"):
        idx = sorted(ionizados[tipo])
        for i, a in enumerate(idx):
            for b in idx[i + 1:]:
                if distancias[a][b] <= distancia_aviso:
                    proximas += 1

    final = mol.GetMol()
    try:
        Chem.SanitizeMol(final)
    except Exception:
        return (None, pka_ini["A"], pka_ini["B"], bitacora, 0,
                "sanitización fallida")

    return (Chem.MolToSmiles(final), pka_ini["A"], pka_ini["B"],
            bitacora, proximas, "ok")


def verificar(smi_antes, smi_despues):
    """Esqueleto conservado y sin pérdida de centros quirales."""
    a = Chem.MolFromSmiles(smi_antes)
    b = Chem.MolFromSmiles(smi_despues) if smi_despues else None
    if b is None:
        return "resultado no válido"
    if a is not None and a.GetNumHeavyAtoms() != b.GetNumHeavyAtoms():
        return "esqueleto alterado"
    if a is not None:
        centros = lambda m: len(Chem.FindMolChiralCenters(
            m, includeUnassigned=False, useLegacyImplementation=False))
        if centros(b) < centros(a):
            return "estereoquímica perdida"
    return "ok"


# ════════════════════════════════════════════════════════════════════════
#  Ejecución
# ════════════════════════════════════════════════════════════════════════
if not PROTONAR:
    print("Protonación desactivada: las estructuras se mantienen neutras.")

elif not MOLGPKA_LISTO:
    df_validas["protonacion"] = "MolGpKa no disponible"
    print("MolGpKa no está disponible: se continúa con las estructuras neutras.")

elif not len(df_validas):
    print("No quedan moléculas que protonar.")

else:
    # Carga única de los modelos.
    modelo_acido = load_model(
        os.path.join(RUTA_MOLGPKA, "models", "weight_acid.pth"), DISPOSITIVO)
    modelo_base = load_model(
        os.path.join(RUTA_MOLGPKA, "models", "weight_base.pth"), DISPOSITIVO)
    print("Modelos cargados una sola vez.")
    print(f"Protonando {len(df_validas)} moléculas "
          f"(pH {PH_OBJETIVO}, margen {MARGEN_MINIMO})...")

    t0 = time.time()
    finales, estados = [], []
    col_acidos, col_basicos, n_sitios, col_proximas = [], [], [], []
    paso = max(1, len(df_validas) // 10)

    for k, fila in enumerate(df_validas.itertuples(index=False)):
        if k and k % paso == 0:
            print(f"  {k}/{len(df_validas)}")

        neutro = fila.SMILES_curado
        candidato, pka_a, pka_b, bitacora, proximas, veredicto = protonar_cascada(
            neutro, modelo_acido, modelo_base, DISPOSITIVO,
            PH_OBJETIVO, MARGEN_MINIMO, DISTANCIA_AVISO,
            MAX_RONDAS_CASCADA)

        col_acidos.append(";".join(f"{v:.2f}" for v in sorted(pka_a)))
        col_basicos.append(";".join(f"{v:.2f}" for v in sorted(pka_b)))
        n_sitios.append(len(pka_a) + len(pka_b))
        col_proximas.append(proximas)

        if veredicto != "ok":
            finales.append(neutro); estados.append(veredicto); continue

        control = verificar(neutro, candidato)
        if control != "ok":
            finales.append(neutro); estados.append(f"revertida: {control}")
        elif not bitacora:
            finales.append(candidato)
            estados.append("sin sitios favorecidos a este pH")
        else:
            finales.append(candidato)
            estados.append(f"protonada ({len(bitacora)} sitio"
                           f"{'s' if len(bitacora) > 1 else ''})")

    df_validas["SMILES_final"]        = finales
    df_validas["protonacion"]         = estados
    df_validas["pka_acidos"]          = col_acidos
    df_validas["pka_basicos"]         = col_basicos
    df_validas["n_sitios_ionizables"] = n_sitios
    df_validas["ionizaciones_proximas"] = col_proximas

    print(f"\nTiempo: {time.time() - t0:.1f} s")
    print("\nResultado de la protonación:")
    print(df_validas["protonacion"].value_counts().to_string())

    cargas = df_validas["SMILES_final"].apply(
        lambda s: (lambda m: Chem.GetFormalCharge(m) if m else None)(
            Chem.MolFromSmiles(s)))
    print("\nDistribución de carga formal:")
    print(cargas.value_counts().sort_index().to_string())

    # pKa cerca del pH: carga incierta.
    def _cerca_del_umbral(fila, margen=1.0):
        crudo = f"{fila.pka_acidos};{fila.pka_basicos}"
        valores = [float(v) for v in crudo.split(";") if v not in ("", "nan")]
        return any(abs(v - PH_OBJETIVO) <= margen for v in valores)

    inciertas = df_validas[df_validas.apply(_cerca_del_umbral, axis=1)]
    print(f"\nMoléculas con algún pKa predicho a menos de 1 unidad del pH de "
          f"trabajo: {len(inciertas)}")
    print("Su estado de carga es incierto; conviene revisarlas de forma "
          "individual si aparecen entre los mejores resultados.")

    n_prox = int((df_validas["ionizaciones_proximas"] > 0).sum())
    if n_prox:
        print(f"\nAVISO: {n_prox} moléculas quedaron con dos o más ionizaciones "
              f"del mismo signo a {DISTANCIA_AVISO} enlaces o menos.")
        print("Este procedimiento no corrige el acoplamiento entre sitios "
              "(ver sección 3.3), de modo que esas especies pueden no existir "
              "al pH indicado. Conviene revisarlas de forma individual.")
        display(df_validas[df_validas["ionizaciones_proximas"] > 0]
                [["ID", "SMILES_final", "ionizaciones_proximas"]].head(10))

    cambiadas = df_validas[df_validas["protonacion"].str.startswith("protonada")]
    if len(cambiadas):
        print(f"\nEjemplos de estructuras modificadas ({len(cambiadas)} en total):")
        display(cambiadas[["ID", "SMILES_curado", "SMILES_final"]].head(5))

Modelos cargados una sola vez.
Protonando 92 moléculas (pH 7.4, margen 1.0)...
  9/92
  18/92
  27/92
  36/92
  45/92
  54/92
  63/92
  72/92
  81/92
  90/92

Tiempo: 12.1 s

Resultado de la protonación:
protonacion
sin sitios favorecidos a este pH    42
protonada (1 sitio)                 41
protonada (2 sitios)                 9

Distribución de carga formal:
SMILES_final
-2     4
-1    23
 0    39
 1    22
 2     4

Moléculas con algún pKa predicho a menos de 1 unidad del pH de trabajo: 27
Su estado de carga es incierto; conviene revisarlas de forma individual si aparecen entre los mejores resultados.

AVISO: 1 moléculas quedaron con dos o más ionizaciones del mismo signo a 4 enlaces o menos.
Este procedimiento no corrige el acoplamiento entre sitios (ver sección 3.3), de modo que esas especies pueden no existir al pH indicado. Conviene revisarlas de forma individual.


,ID,SMILES_final,ionizaciones_proximas
90,170119,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)([O-])OP(...,1



Ejemplos de estructuras modificadas (50 en total):


,ID,SMILES_curado,SMILES_final
0,448219,CCCC[C@H](N)C(O)(O)CNC1CC1,CCCC[C@H]([NH3+])C(O)(O)CNC1CC1
1,49866822,NC[C@@]1(O)NC(=O)C(=O)N1CC(=O)O,[NH3+]C[C@@]1(O)NC(=O)C(=O)N1CC(=O)[O-]
2,93485622,C/C(N)=N\CC[C@H](F)C[C@H](N)C(=O)O,C/C(N)=[NH+]\CC[C@H](F)C[C@H](N)C(=O)[O-]
5,24748045,O=C(O)[C@@H](CS)CCCc1ccccc1,O=C([O-])[C@@H](CS)CCCc1ccccc1
6,10062718,O=C1C=C(c2cccc(O)c2)S(=O)(=O)N1,O=C1C=C(c2cccc(O)c2)S(=O)(=O)[N-]1


<a id="s41"></a>
# 4. Geometría tridimensional

## 4.1 Fundamento

Hasta aquí la molécula es un **grafo**: átomos y enlaces, sin coordenadas. El docking
necesita posiciones (x, y, z).

### Embedding con ETKDG

**ETKDG** (*Experimental-Torsion Knowledge Distance Geometry*) trabaja en tres etapas:

1. **Matriz de distancias**
2. **Coordenadas compatibles**
3. **Corrección experimental**

Se generan **varias conformaciones** porque cada intento converge a un mínimo local
distinto, se conserva la de menor energía.

### Minimización con MMFF94

Un **campo de fuerza** asigna energía a cada geometría tratando la molécula como un
sistema mecánico clásico. **MMFF94** está parametrizado para
moléculas orgánicas pequeñas de interés farmacéutico.

$$E_{\text{total}} = E_{\text{enlace}} + E_{\text{ángulo}} + E_{\text{torsión}}
+ E_{\text{fuera de plano}} + E_{\text{vdW}} + E_{\text{electrostática}}$$

- $E_{\text{enlace}}$ — costo de estirar o comprimir un enlace respecto a su longitud de equilibrio.
- $E_{\text{ángulo}}$ — costo de abrir o cerrar el ángulo entre tres átomos enlazados (≈109.5° para sp³, 120° para sp²).
- $E_{\text{torsión}}$ — energía del giro alrededor de enlaces sencillos; mínimos en conformaciones alternadas.
- $E_{\text{fuera de plano}}$ — penaliza que un átomo que debe ser plano se salga de su plano.
- $E_{\text{vdW}}$ — entre átomos **no** enlazados: repulsión fuerte a corta distancia, atracción débil a media.
- $E_{\text{electrostática}}$ — entre cargas parciales. **Este es el término que cambia al activar la protonación**.

La minimización mueve los átomos siguiendo las fuerzas hasta llegar a un **mínimo local**
(el más cercano al punto de partida, no el global).

<a id="s42"></a>
## 4.2 Ejecución

**Qué hace:** añade hidrógenos explícitos (necesarios porque participan en puentes de
hidrógeno y choques estéricos), genera `N_CONFORMEROS_PRUEBA` conformaciones, minimiza
cada una con MMFF94 y conserva la de menor energía.

Mide la energía **antes** de minimizar y **después**; ambas van al reporte y su diferencia
indica cuánta tensión tenía la geometría inicial.

También se registra la **carga formal** de cada estructura entregada: cero en casi todas,
distinta de cero solo donde la neutralización no era químicamente posible.

In [28]:
#@title Generar y minimizar la geometría 3D { display-mode: "form" }
from rdkit.Chem import AllChem

def _geometria_una(args):
    """Varias conformaciones, minimizadas con MMFF94; devuelve la de menor energía."""
    mol_id, smiles, n_confs, semilla, max_iter = args
    r = {"ID": mol_id, "molblock": None, "energia_inicial": None,
         "energia_final": None, "estado_3d": "ok", "motivo_3d": ""}

    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            r["estado_3d"] = "fallida"; r["motivo_3d"] = "SMILES ilegible"
            return r

        mol = Chem.AddHs(mol)   # explícitos: necesarios para el 3D

        params = AllChem.ETKDGv3()
        params.randomSeed              = semilla
        params.enforceChirality        = True
        params.useExpTorsionAnglePrefs = True
        params.useBasicKnowledge       = True
        params.useMacrocycleTorsions   = True
        params.useSmallRingTorsions    = True
        params.numThreads              = 1   # el paralelismo va por molécula
        params.maxIterations           = 1000

        ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
        if len(ids) == 0:
            # Último recurso.
            params.useRandomCoords = True
            ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
        if len(ids) == 0:
            r["estado_3d"] = "fallida"
            r["motivo_3d"] = "no se pudo generar ninguna conformación"
            return r

        props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94")
        if props is None:
            r["estado_3d"] = "fallida"
            r["motivo_3d"] = "MMFF94 no tiene parámetros para estos átomos"
            return r

        mejor_id, mejor_E, E_inicial_mejor = None, None, None
        for cid in ids:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            E_ini = ff.CalcEnergy()          # antes de minimizar
            ff.Minimize(maxIts=max_iter)     # una sola vez
            E_fin = ff.CalcEnergy()          # después de minimizar
            if mejor_E is None or E_fin < mejor_E:
                mejor_E, mejor_id, E_inicial_mejor = E_fin, cid, E_ini

        # Solo la de menor energía.
        mol_final = Chem.Mol(mol)
        mol_final.RemoveAllConformers()
        mol_final.AddConformer(mol.GetConformer(mejor_id), assignId=True)
        mol_final.SetProp("_Name", str(mol_id))

        r["molblock"]        = Chem.MolToMolBlock(mol_final)
        r["energia_inicial"] = round(E_inicial_mejor, 3)
        r["energia_final"]   = round(mejor_E, 3)
        return r

    except Exception as e:
        r["estado_3d"] = "fallida"
        r["motivo_3d"] = f"{type(e).__name__}: {str(e)[:80]}"
        return r


# SMILES_final lo define la sección 3; si no se ejecutó, se usa la curada.
if "SMILES_final" not in df_validas.columns:
    df_validas["SMILES_final"] = df_validas["SMILES_curado"]
    df_validas["protonacion"]  = "sección 3 no ejecutada"

tareas = [(r.ID, r.SMILES_final, N_CONFORMEROS_PRUEBA,
           SEMILLA_ALEATORIA, MAX_ITER_MMFF)
          for r in df_validas.itertuples(index=False)]

print(f"Generando geometría 3D de {len(tareas)} moléculas "
      f"({N_CONFORMEROS_PRUEBA} conformaciones cada una, {N_NUCLEOS} núcleos)...")
t0 = time.time()

if N_NUCLEOS > 1:
    with Pool(N_NUCLEOS) as pool:
        res_3d = pool.map(_geometria_una, tareas, chunksize=5)
else:
    res_3d = [_geometria_una(t) for t in tareas]

print(f"Tiempo: {time.time() - t0:.1f} s")

# Carga formal de la estructura entregada.
def _carga_formal(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.GetFormalCharge(m) if m is not None else None

df_validas["carga_formal"] = df_validas["SMILES_final"].apply(_carga_formal)

df_3d = pd.DataFrame(res_3d)
df_final = df_validas.merge(df_3d, on="ID", how="left")

n_ok3d = int((df_final["estado_3d"] == "ok").sum())
print(f"\nGeometría 3D obtenida : {n_ok3d}")
print(f"Fallidas              : {len(df_final) - n_ok3d}")

if len(df_final) - n_ok3d:
    print("\nMotivos de fallo:")
    print(df_final.loc[df_final["estado_3d"] != "ok", "motivo_3d"]
          .value_counts().to_string())

n_cargadas = int((df_final["carga_formal"].fillna(0) != 0).sum())
print(f"\nEstructuras con carga formal distinta de cero: {n_cargadas} "
      f"(el resto se entrega neutra)")

Generando geometría 3D de 92 moléculas (10 conformaciones cada una, 1 núcleos)...
Tiempo: 133.1 s

Geometría 3D obtenida : 92
Fallidas              : 0

Estructuras con carga formal distinta de cero: 53 (el resto se entrega neutra)


<a id="s51"></a>
# 5. Resultados

## 5.1 Control visual

In [29]:
#@title Ver algunas estructuras en 3D { display-mode: "form" }
import py3Dmol
from IPython.display import display, HTML

N_A_MOSTRAR = 3   #@param {type:"integer"}

muestra = df_final[df_final["estado_3d"] == "ok"].head(N_A_MOSTRAR)

for fila in muestra.itertuples(index=False):
    display(HTML(f"<h4>{fila.ID} &nbsp;—&nbsp; "
                 f"E<sub>final</sub> = {fila.energia_final} kcal/mol &nbsp;|&nbsp; "
                 f"PM = {fila.PM} Da</h4>"))
    visor = py3Dmol.view(width=420, height=320)
    visor.addModel(fila.molblock, "mol")
    visor.setStyle({"stick": {}})
    visor.setBackgroundColor("white")
    visor.zoomTo()
    visor.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [30]:
#@title Rejilla 2D de las estructuras finales { display-mode: "form" }
import mols2grid

ok = df_final[df_final["estado_3d"] == "ok"].copy()
if len(ok):
    ok["Mol"] = ok["SMILES_final"].apply(Chem.MolFromSmiles)
    columnas = ["ID", "PM", "SA_score", "alerta_tox"]
    columnas = [c for c in columnas if c in ok.columns]
    display(mols2grid.display(ok.head(48), mol_col="Mol",
                              subset=["img"] + columnas, size=(190, 160)))
else:
    print("No hay estructuras para mostrar.")

<a id="s52"></a>
## 5.2 Empaquetar

**Por qué SDF:** guarda coordenadas 3D, orden de enlace y cargas formales en un solo archivo de texto, y admite varias moléculas en el mismo archivo, lo cual es ideal para realizar un cribado mediante acoplamiento molecular (docking).

**Contenido del ZIP:**

```text
ligandos_docking.zip
├── sdf_individuales/      un .sdf por molécula, nombrado con su ID
├── ligandos_todos.sdf     todas en un solo archivo
├── ligandos_finales.csv   ← reporte principal
├── descartadas.csv        las que no llegaron, con etapa y motivo
├── duplicados.csv         pares de IDs idénticos
└── parametros.json        la configuración exacta de esta corrida
```

`ligandos_finales.csv` contiene **solo las moléculas que sí obtuvieron geometría 3D**: es el índice del contenido del ZIP, cada fila corresponde a un `.sdf` que existe. Las moléculas descartadas van a `descartadas.csv` con el motivo documentado.

In [31]:
#@title Empaquetar los resultados { display-mode: "form" }
import os, json, shutil, zipfile
from datetime import datetime

CARPETA = NOMBRE_SALIDA
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(os.path.join(CARPETA, "sdf_individuales"), exist_ok=True)

exitosas = df_final[df_final["estado_3d"] == "ok"].copy()

# 1. Archivos SDF
ruta_sdf_unico = os.path.join(CARPETA, "ligandos_todos.sdf")
escritor = Chem.SDWriter(ruta_sdf_unico) if GUARDAR_SDF_UNICO else None

for fila in exitosas.itertuples(index=False):
    mol = Chem.MolFromMolBlock(fila.molblock, removeHs=False)
    if mol is None:
        continue
    mol.SetProp("_Name", str(fila.ID))
    # Las propiedades viajan dentro del SDF.
    mol.SetProp("ID", str(fila.ID))
    mol.SetProp("PM", str(fila.PM))
    mol.SetProp("SMILES_final", str(fila.SMILES_final))
    mol.SetProp("carga_formal", str(fila.carga_formal))
    mol.SetProp("energia_MMFF", str(fila.energia_final))
    if "SA_score" in exitosas.columns and fila.SA_score is not None:
        mol.SetProp("SA_score", str(fila.SA_score))
    if "alerta_tox" in exitosas.columns:
        mol.SetProp("alerta_tox", str(fila.alerta_tox))

    if GUARDAR_SDF_INDIVIDUALES:
        nombre_seguro = "".join(c if c.isalnum() or c in "-_." else "_"
                                for c in str(fila.ID))
        w = Chem.SDWriter(os.path.join(CARPETA, "sdf_individuales",
                                       f"{nombre_seguro}.sdf"))
        w.write(mol); w.close()

    if escritor is not None:
        escritor.write(mol)

if escritor is not None:
    escritor.close()

# 2. CSV final: solo las moléculas con estructura 3D
columnas_csv = ["ID", "SMILES_original", "SMILES_curado", "SMILES_final",
                "QED", "SA_score",
                "PM", "logP", "TPSA", "donadores_H", "aceptores_H",
                "enlaces_rotables", "anillos_aromaticos",
                "alerta_tox", "n_alertas", "alerta_detalle",
                "carga_formal", "ionizaciones_proximas",
                "estereo_indefinida", "elementos_no_usuales",
                "energia_inicial", "energia_final"]
columnas_csv = [c for c in columnas_csv if c in exitosas.columns]
exitosas[columnas_csv].to_csv(
    os.path.join(CARPETA, "ligandos_finales.csv"), index=False)

# 3. Registro de lo descartado
descartes = []
no_curadas = df_curado[df_curado["estado"] != "ok"][["ID", "SMILES_original", "motivo"]]
descartes.append(no_curadas.assign(etapa="curación"))

if len(df_duplicados):
    descartes.append(pd.DataFrame({
        "ID": df_duplicados["ID_eliminado"],
        "SMILES_original": "",
        "motivo": "duplicado de " + df_duplicados["ID_conservado"],
        "etapa": "duplicados"}))

if FILTRO_PESO_MOLECULAR and len(fuera):
    descartes.append(fuera[["ID", "SMILES_original", "motivo"]].assign(etapa="peso molecular"))

fallidas_3d = df_final[df_final["estado_3d"] != "ok"]
if len(fallidas_3d):
    descartes.append(pd.DataFrame({
        "ID": fallidas_3d["ID"],
        "SMILES_original": fallidas_3d["SMILES_original"],
        "motivo": fallidas_3d["motivo_3d"],
        "etapa": "geometría 3D"}))

df_descartadas = (pd.concat(descartes, ignore_index=True)
                  if descartes else pd.DataFrame(columns=["ID","SMILES_original","motivo","etapa"]))
df_descartadas.to_csv(os.path.join(CARPETA, "descartadas.csv"), index=False)
df_duplicados.to_csv(os.path.join(CARPETA, "duplicados.csv"), index=False)

# 4. Configuración usada
parametros = {
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "moleculas_entrada": int(len(df_entrada)),
    "moleculas_entregadas": int(len(exitosas)),
    "curacion_estandar": CURACION_ESTANDAR,
    "tautomero_canonico": TAUTOMERO_CANONICO,
    "eliminar_duplicados": ELIMINAR_DUPLICADOS,
    "filtro_peso_molecular": FILTRO_PESO_MOLECULAR,
    "pm_minimo": PM_MINIMO, "pm_maximo": PM_MAXIMO,
    "filtro_toxicologico": FILTRO_TOXICOLOGICO,
    "catalogos_alertas": CATALOGOS_ALERTAS,
    "descartar_alertadas": DESCARTAR_ALERTADAS,
    "protonar": PROTONAR,
    "ph_objetivo": PH_OBJETIVO if PROTONAR else None,
    "margen_minimo": MARGEN_MINIMO if PROTONAR else None,
    "distancia_aviso": DISTANCIA_AVISO if PROTONAR else None,
    "motor_protonacion": "MolGpKa en cascada" if PROTONAR else None,
    "n_conformeros_prueba": N_CONFORMEROS_PRUEBA,
    "semilla": SEMILLA_ALEATORIA,
    "max_iter_mmff": MAX_ITER_MMFF,
    "rdkit": rdkit.__version__,
}
with open(os.path.join(CARPETA, "parametros.json"), "w") as f:
    json.dump(parametros, f, indent=2, ensure_ascii=False)

# 5. Comprimir
ruta_zip = shutil.make_archive(NOMBRE_SALIDA, "zip", CARPETA)
tam_mb = os.path.getsize(ruta_zip) / 1e6

print("═" * 62)
print(f"  Entrada            : {len(df_entrada)} moléculas")
print(f"  Entregadas en 3D   : {len(exitosas)} moléculas")
print(f"  Descartadas        : {len(df_descartadas)}")
print("─" * 62)
print(f"  Archivo            : {os.path.basename(ruta_zip)}  ({tam_mb:.2f} MB)")
print("═" * 62)

if len(df_descartadas):
    print("\nDescartes por etapa:")
    print(df_descartadas["etapa"].value_counts().to_string())

══════════════════════════════════════════════════════════════
  Entrada            : 100 moléculas
  Entregadas en 3D   : 92 moléculas
  Descartadas        : 8
──────────────────────────────────────────────────────────────
  Archivo            : Prueba_1.zip  (0.19 MB)
══════════════════════════════════════════════════════════════

Descartes por etapa:
etapa
peso molecular    8


<a id="s53"></a>
## 5.3 Descargar

Las estructuras tridimensionales resultantes se encuentran listas para su evaluación en cualquier programa de acoplamiento molecular (docking).

El campo `ID` se preserva dentro del archivo SDF, por lo que el archivo de salida del acoplamiento molecular podrá correlacionarse fácilmente con el reporte `ligandos_finales.csv` utilizando esta columna.

In [32]:
#@title Descargar el ZIP { display-mode: "form" }
try:
    from google.colab import files
    files.download(ruta_zip)
    print("Descarga iniciada.")
except ImportError:
    print(f"Fuera de Colab. El archivo está en: {os.path.abspath(ruta_zip)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada.


---

| Etapa | Resolución analítica |
|---|---|
| Curación | Remoción de sales, metales y unificación a un estado de carga neutro. |
| Duplicados | Eliminación de registros repetidos para optimizar recursos computacionales. |
| Peso molecular | Restricción del análisis a entidades químicas con dimensiones aptas para interacciones específicas. |
| Alertas | Identificación y etiquetado sistemático de subestructuras vinculadas a reactividad inespecífica. |
| SA score | Cuantificación de la accesibilidad sintética para facilitar el cribado posterior al acoplamiento molecular. |
| Protonación (MolGpKa) | Predicción del pKa de cada sitio y asignación secuencial del estado de carga al pH indicado. |
| ETKDG + MMFF94 | Generación y minimización energética del espacio conformacional tridimensional. |

Si la protonación se mantuvo desactivada, **las estructuras salen neutras**.
En caso de presentarse inconvenientes en la ejecución, se aconseja consultar el archivo `descartadas.csv`. En este registro, la columna `etapa` identifica el paso metodológico donde el procesamiento fue interrumpido y la columna `motivo` especifica la causa estructural que impidió la consecución del análisis.

---

QFB. Luis Ángel Urdez Rojas

## Referencias y Citas

Este flujo de trabajo computacional forma parte del protocolo metodológico desarrollado en la tesis listada a continuación. En caso de utilizar o adaptar este flujo de trabajo en investigaciones futuras, se debe citar el trabajo original de los autores, así como los enlaces de los artículos de las herramientas implementadas.

### Tesis asociada
* Franco Chávez, B., & Urdez Rojas, L. Á. (2026). *IDENTIFICACIÓN in silico DE POTENCIALES INHIBIDORES DE MRP4/ABCC4 PARA EL TRATAMIENTO DE LA LEUCEMIA LINFOBLÁSTICA AGUDA* [Tesis de Licenciatura, Universidad Autónoma de Querétaro]. Director de tesis: Dr. Rubén Antonio Romo Mancillas. Repositorio Institucional UAQ. URI: [https://ri-ng.uaq.mx/handle/123456789/13105](https://ri-ng.uaq.mx/handle/123456789/13105)

### Predicción de pKa (MolGpKa)
* Li, X., et al. (2021). MolGpKa: A Web Server for Small Molecule pKa Prediction Using a Graph-Convolutional Neural Network. *Journal of Chemical Information and Modeling*.
* Código fuente y repositorio: [https://github.com/Xundrug/MolGpKa](https://github.com/Xundrug/MolGpKa)

### Motor quimioinformático (RDKit)
* RDKit: Open-source cheminformatics. [https://www.rdkit.org](https://www.rdkit.org)

### Algoritmos de Geometría y Energía
* **ETKDG (Generación de conformaciones):** Wang, S., Witek, J., Landrum, G. A., & Riniker, S. (2020). Improving conformer generation for small rings and macrocycles based on distance geometry and experimental torsional-angle preferences. *Journal of Chemical Information and Modeling*, 60(4), 2044-2058.
* **MMFF94 (Campo de fuerza):** Halgren, T. A. (1996). Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94. *Journal of Computational Chemistry*, 17(5-6), 490-519.

### Filtros y Descriptores
* **SA Score:** Ertl, P., & Schuffenhauer, A. (2009). Estimation of synthetic accessibility score of drug-like molecules based on molecular complexity and fragment contributions. *Journal of Cheminformatics*, 1(1), 1-11.
* **PAINS (Pan Assay Interference Compounds):** Baell, J. B., & Holloway, G. A. (2010). New substructure filters for removal of pan assay interference compounds (PAINS) from screening libraries and for their exclusion in bioassays. *Journal of Medicinal Chemistry*, 53(7), 2719-2740.